# Day 053 — Exercise 4: CORS — Let the Browser In

**What you'll build:** `add_cors(app, origins)` — enable Cross-Origin Resource Sharing on the backend so a browser front-end served from a different origin can call the API.

**Why it matters:** Your Python tests can call the backend freely, but a *browser* can't. The same-origin policy blocks JavaScript on `http://localhost:8501` from calling `http://localhost:8000` unless the server sends `access-control-allow-origin`. This is the #1 'it works in my script but not in the browser' bug — and one middleware fixes it.

## Provided: Setup + Backend

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import httpx
import ollama


# ---- The AI backend (built on Day 52 — provided here) ----
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    reply: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app

## Your Implementation

In [ ]:
def add_cors(app: FastAPI, origins: list) -> FastAPI:
    """
    Add CORSMiddleware allowing the given origins. Return the same app.
    Allow all methods and headers; allow credentials.
    """
    # TODO: app.add_middleware(
    #     CORSMiddleware,
    #     allow_origins=origins,
    #     allow_credentials=True,
    #     allow_methods=['*'],
    #     allow_headers=['*'],
    # )
    # TODO: return app
    pass

## Check Your Work

In [ ]:
ALLOWED = 'http://localhost:8501'


def _make_app():
    app = FastAPI()
    @app.get('/health')
    def _h():
        return {'status': 'ok'}
    return app


def _run_checks():
    total = 5
    passed = 0

    # Check 1: add_cors returns the app (a FastAPI instance)
    try:
        app = add_cors(_make_app(), [ALLOWED])
        assert isinstance(app, FastAPI), 'add_cors must return the app'
        client = TestClient(app)
        passed += 1; print('✅ Check 1: add_cors returns the FastAPI app')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: allowed origin gets the access-control-allow-origin header
    try:
        r = client.get('/health', headers={'Origin': ALLOWED})
        assert r.headers.get('access-control-allow-origin') == ALLOWED, \
            f"missing/wrong ACAO header: {r.headers.get('access-control-allow-origin')}"
        passed += 1; print('✅ Check 2: allowed origin -> ACAO header set')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: preflight OPTIONS is answered with 200 + ACAO
    try:
        pre = client.options('/health', headers={
            'Origin': ALLOWED,
            'Access-Control-Request-Method': 'GET',
        })
        assert pre.status_code == 200, f'preflight status {pre.status_code}'
        assert pre.headers.get('access-control-allow-origin') == ALLOWED
        passed += 1; print('✅ Check 3: preflight OPTIONS handled (200 + ACAO)')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: a normal request (no Origin) still works
    try:
        r = client.get('/health')
        assert r.status_code == 200 and r.json()['status'] == 'ok'
        passed += 1; print('✅ Check 4: same-origin request still works')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: a disallowed origin does NOT get an allow header for itself
    try:
        r = client.get('/health', headers={'Origin': 'http://evil.example'})
        assert r.headers.get('access-control-allow-origin') != 'http://evil.example', \
            'disallowed origin must not be echoed as allowed'
        passed += 1; print('✅ Check 5: disallowed origin is not allowed')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def add_cors(app: FastAPI, origins: list) -> FastAPI:
    """Enable CORS so a browser front-end on a DIFFERENT origin can call this API.

    A browser enforces the same-origin policy: JavaScript on http://localhost:8501
    may not call http://localhost:8000 unless the server opts in with CORS
    headers. CORSMiddleware adds the `access-control-allow-origin` header (and
    answers preflight OPTIONS requests) for the origins you allow.
    """
    app.add_middleware(
        CORSMiddleware,
        allow_origins=origins,
        allow_credentials=True,
        allow_methods=['*'],
        allow_headers=['*'],
    )
    return app
```

**Why this works:** `CORSMiddleware` intercepts responses and, when the request's `Origin` is in `allow_origins`, adds the `access-control-allow-origin` header the browser demands. It also answers the preflight `OPTIONS` request browsers send before a real cross-origin POST. Note this is a *browser* mechanism — `TestClient` and `httpx` ignore it, which is why your notebook tests worked without CORS but the deployed browser app needs it. `backend.py` calls `add_cors(app, ['http://localhost:8501'])`.
</details>